# 06 - Strands Agent
Define a Strands agent with custom @tool decorators, BedrockModel, and run it locally with graph and KB tools.

## Step 1: Load Configuration

In [ ]:
import boto3
import json
from pathlib import Path

config = json.loads(Path("config.json").read_text())
REGION             = config.get("REGION", "us-east-1")
PREFIX             = config.get("PROJECT_PREFIX", "orgiq")
MODEL_ID           = config.get("MODEL_ID", "amazon.nova-pro-v1:0")
AURORA_CLUSTER_ARN = config["AURORA_CLUSTER_ARN"]
AURORA_SECRET_ARN  = config["AURORA_SECRET_ARN"]
DB_NAME            = config.get("DB_NAME", "orgiq")
GRAPH_NAME         = config.get("GRAPH_NAME", "orgiq_graph")

rds_data = boto3.client("rds-data", region_name=REGION)


def _cypher(query):
    sql = f"SELECT * FROM ag_catalog.cypher(\'{GRAPH_NAME}\', $$ {query} $$) as (result ag_catalog.agtype)"
    resp = rds_data.execute_statement(
        resourceArn=AURORA_CLUSTER_ARN,
        secretArn=AURORA_SECRET_ARN,
        database=DB_NAME,
        sql=sql,
    )
    return [[list(f.values())[0] for f in row if f] for row in resp.get("records", [])]


print(f"Region: {REGION}  |  DB: {DB_NAME}  |  Graph: {GRAPH_NAME}")


## Step 2: Define Custom Tools

Create @tool-decorated functions that the agent can call.

In [ ]:
from strands import Agent, tool
from strands.models.bedrock import BedrockModel

bedrock_agent_runtime = boto3.client("bedrock-agent-runtime", region_name=REGION)


@tool
def search_experts(skill: str, limit: int = 5) -> str:
    """Search the knowledge graph for people with expertise in a given skill or domain.

    Args:
        skill: The skill or domain to search for (e.g. 'Kubernetes', 'ML', 'security')
        limit: Maximum number of results to return (default 5)
    """
    rows = _cypher(
        f"MATCH (p:Person)-[r:HAS_SKILL]->(s:Skill) "
        f"WHERE toLower(s.name) CONTAINS toLower(\'{skill}\') "
        f"RETURN p.name, p.role, s.name, coalesce(r.weight, 1.0) as strength "
        f"ORDER BY strength DESC LIMIT {limit}"
    )
    results = [
        {"name": str(r[0]), "role": str(r[1]), "skill": str(r[2]), "strength": float(r[3])}
        for r in rows
    ]
    return json.dumps(results, ensure_ascii=False)


In [ ]:
@tool
def search_documents(query: str, num_results: int = 3) -> str:
    """Search the knowledge base for relevant documents using RAG retrieval.
    
    Args:
        query: Natural language search query
        num_results: Number of results to return
    """
    response = bedrock_agent_runtime.retrieve(
        knowledgeBaseId=KB_ID,
        retrievalQuery={"text": query},
        retrievalConfiguration={
            "vectorSearchConfiguration": {"numberOfResults": num_results}
        }
    )
    
    if not response["retrievalResults"]:
        return "No relevant documents found."
    
    output = []
    for i, result in enumerate(response["retrievalResults"], 1):
        score = result.get("score", 0)
        text = result["content"]["text"][:300]
        output.append(f"[{i}] (score: {score:.3f}) {text}")
    return "\n\n".join(output)


print("\u2713 search_documents tool defined")

In [ ]:
from strands import Agent, tool
from strands.models.bedrock import BedrockModel

bedrock_agent_runtime = boto3.client("bedrock-agent-runtime", region_name=REGION)


@tool
def search_experts(skill: str, limit: int = 5) -> str:
    """Search the knowledge graph for people with expertise in a given skill or domain.

    Args:
        skill: The skill or domain to search for (e.g. 'Kubernetes', 'ML', 'security')
        limit: Maximum number of results to return (default 5)
    """
    rows = _cypher(
        f"MATCH (p:Person)-[r:HAS_SKILL]->(s:Skill) "
        f"WHERE toLower(s.name) CONTAINS toLower(\'{skill}\') "
        f"RETURN p.name, p.role, s.name, coalesce(r.weight, 1.0) as strength "
        f"ORDER BY strength DESC LIMIT {limit}"
    )
    results = [
        {"name": str(r[0]), "role": str(r[1]), "skill": str(r[2]), "strength": float(r[3])}
        for r in rows
    ]
    return json.dumps(results, ensure_ascii=False)


## Step 3: Create the Strands Agent

Wire up BedrockModel with our tools and system prompt.

In [ ]:
# Create the Bedrock model
model = BedrockModel(
    model_id=MODEL_ID,
    region_name=REGION
)

# Define the system prompt
SYSTEM_PROMPT = """You are OrgIQ Search, an organizational intelligence assistant.
You help users find experts, documents, and knowledge within the organization.

You have access to:
- A knowledge graph with people, topics, and documents
- A document knowledge base with full-text search

When answering questions:
1. Use search_experts to find people with relevant expertise
2. Use search_documents to find relevant documentation
3. Use get_person_profile for detailed information about specific people
4. Synthesize results into clear, actionable answers
5. Always cite your sources (person names, document titles)
"""

# Create the agent
agent = Agent(
    model=model,
    system_prompt=SYSTEM_PROMPT,
    tools=[search_experts, search_documents, get_person_profile]
)

print("\u2713 Strands Agent created")
print(f"  Model: {MODEL_ID}")
print(f"  Tools: search_experts, search_documents, get_person_profile")

## Step 4: Test - Expert Search Query

In [ ]:
# Test the agent with an expert search query
response = agent("Who are the best Kubernetes experts in our organization?")
print("Agent response:")
print(response)

## Step 5: Test - Document Search Query

In [ ]:
# Test document search
response = agent("What do we know about zero trust authentication? Summarize our documentation.")
print("Agent response:")
print(response)

## Step 6: Test - Person Profile Query

In [ ]:
# Test person profile lookup
response = agent("Tell me about Alice Chen - what does she work on and what has she written?")
print("Agent response:")
print(response)

## Step 7: Test - Multi-Tool Complex Query

In [ ]:
# Complex query requiring multiple tools
response = agent(
    "I need to set up a new ML pipeline. Who should I talk to, "
    "and what existing documentation can help me get started?"
)
print("Agent response:")
print(response)

## Step 8: Agent with Guardrails

Create a version of the agent with guardrails applied.

In [ ]:
# Create a guarded model
guarded_model = BedrockModel(
    model_id=MODEL_ID,
    region_name=REGION,
    guardrail_config={
        "guardrailIdentifier": GUARDRAIL_ID,
        "guardrailVersion": GUARDRAIL_VERSION
    }
)

guarded_agent = Agent(
    model=guarded_model,
    system_prompt=SYSTEM_PROMPT,
    tools=[search_experts, search_documents, get_person_profile]
)

print("\u2713 Guarded agent created")

# Test with safe query
print("\n--- Safe query ---")
response = guarded_agent("Who knows about CI/CD?")
print(response)

In [ ]:
# Test with query that should be blocked
print("--- Blocked query (salary) ---")
try:
    response = guarded_agent("What is Alice's salary?")
    print(response)
except Exception as e:
    print(f"\u2713 Guardrail blocked the request: {e}")

## Step 9: Streaming Agent Response

In [ ]:
# Use streaming for real-time output
import sys

print("Streaming response:")
print("-" * 40)

result = agent.stream("Give me a brief overview of our security expertise.")
for event in result:
    if hasattr(event, "data"):
        sys.stdout.write(event.data)
        sys.stdout.flush()

print("\n" + "-" * 40)
print("\u2713 Streaming complete")

## Step 10: Agent Conversation History

In [ ]:
# Multi-turn conversation
conversation_agent = Agent(
    model=model,
    system_prompt=SYSTEM_PROMPT,
    tools=[search_experts, search_documents, get_person_profile]
)

# Turn 1
print("User: Who knows about authentication?")
response1 = conversation_agent("Who knows about authentication?")
print(f"Agent: {response1}\n")

# Turn 2 - referencing previous context
print("User: What documents have they written?")
response2 = conversation_agent("What documents have they written?")
print(f"Agent: {response2}\n")

print("\u2713 Multi-turn conversation working")

## Summary

- [x] Three custom @tool functions defined (search_experts, search_documents, get_person_profile)
- [x] Strands Agent created with BedrockModel (Claude Sonnet)
- [x] Expert search queries working via Aurora knowledge graph
- [x] Document search working via Bedrock KB RAG
- [x] Person profile lookups working
- [x] Multi-tool complex queries handled
- [x] Guardrail integration verified (safe pass, unsafe block)
- [x] Streaming responses working
- [x] Multi-turn conversation context maintained

**Next:** [07-agentcore-memory.ipynb](07-agentcore-memory.ipynb) — Add persistent memory to the agent with AgentCore Memory